In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_spleen")

Prepare data

In [3]:
rna = sc.read_h5ad("rna.h5ad")
protein = sc.read_h5ad("protein.h5ad")

In [4]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [5]:
rna.var_names_make_unique()
rna.obs_names_make_unique()
protein.var_names_make_unique()
protein.obs_names_make_unique()

In [6]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(protein, output_file="protein.h5")

In [7]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [8]:
# with h5py.File(data_folder+"atac.h5", "r") as f:
    # print(list(f["matrix"]))

In [9]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_spleen/"
# bm.run(methods=bm.multi_omics_methods,
#        RNA_file_path=data_folder+"rna.h5",
#        ADT_file_path=data_folder+"/protein.h5",
#        n_cluster=3,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/")

In [10]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/rna.csv", index_col=0)

In [11]:
# if "cluster" in rna.obs: del rna.obs["cluster"]
# if "cluster_colors" in rna.uns: del rna.uns["cluster_colors"]
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [12]:
# sc.pl.umap(rna, color="cluster")
# sc.pl.spatial(rna, color="cluster", spot_size=1.5)

Plot

In [13]:
# methods =  ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",    "scMM",       "scMDC",
#                 "Matilda",   "moETM",  "MISO",   "SpatialGlue",  "COSMOS",     "PRESENT",
#                 "SMOPCA",       "CellCharter", "spaMultiVAE",      "TotalVI",  "sciPENN", "rna", "adt"
#                ]
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

In [14]:
# res_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/"
# def search_resolution(adata, fixed_clus_count, increment=0.001):
#     closest_count = np.inf  
#     closest_res = None  
    
#     for res in sorted(list(np.arange(0.01, 0.2, increment)), reverse=False):
#         sc.tl.leiden(adata, random_state=0, resolution=res, key_added="temp_label")
#         print(len(set(adata.obs["temp_label"])))
#         count_unique_leiden = len(list(set(adata.obs["temp_label"])))
#         current_diff = abs(count_unique_leiden - fixed_clus_count)
#         if current_diff < closest_count:
#             closest_count = current_diff
#             closest_res = res
#         if count_unique_leiden == fixed_clus_count:
#             break

#     return closest_res
# def recluster(methods, ncluster):
#     df = pd.read_csv(f"{res_dir}/{methods[0].lower()}_latent.csv", index_col=0)
#     adata = sc.AnnData(X=np.zeros((df.shape[0], 10)))
#     for m in methods:
#         adata.obsm[f"X_{m}"] = np.array(pd.read_csv(f"{res_dir}/{m.lower()}_latent.csv", index_col=0))
#         sc.pp.neighbors(adata, use_rep=f"X_{m}")
#         sc.tl.umap(adata)
#         res = search_resolution(adata, fixed_clus_count=ncluster)
#         sc.tl.leiden(adata, resolution=res, key_added="cluster")
#         print(m, len(set(adata.obs["cluster"])))
#         umap = pd.DataFrame(adata.obsm["X_umap"], columns=["UMAP1", "UMAP2"], index=adata.obs_names)
#         umap.insert(2, "cluster", adata.obs['cluster'].values)
#         umap.to_csv(os.path.join(res_dir, m.lower() + ".csv"))

In [15]:
# recluster([ "scMDC"], ncluster=3)

Plot

In [15]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/"
methods = ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",   "scMDC",    "scMM",
"Matilda",   "moETM",    "TotalVI",  "sciPENN",  "SpatialGlue",  "COSMOS",
"MISO",  "PRESENT",  "spaMultiVAE",  "SMOPCA",   "CellCharter"]
res = bm.read_result(path=result_folder,
                     methods=methods + ["rna", "adt"],
                     reindex=False)

2026-04-03 19:13:05 - WARNING - '_latent' result for 'Seurat_WNN' not found at: /mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_spleen/seurat_wnn_latent.csv


In [16]:
from benchmarker import  read_sparse_h5, recompute_aggregate_scores

In [17]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [rna.obsm["spatial"]]
spatial = transform_coord(spatial, vertical=False, horizontal=True, angle=0)

In [18]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO", "spaMultiVAE"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"

In [19]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Mouse_spleen"

In [20]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [22]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(1.97, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=10,
#                 ncol=1,
#                 xlabel=["RNA", "Protein"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "adt"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_single_modal.pdf",
#                 rasterized=True
#                 )

In [22]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(16, 6.8),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=10,
#                 ncol=8,
#                 xlabel=["RNA", "Protein", "Seurat_WNN", "MOFA2", "sciPENN", "moETM", "Matilda", "scMDC", "scMM",
#                 "MultiVI", "Multigrate", "TotalVI",
#                 "SMOPCA", "PRESENT", "SpatialGlue", "CellCharter", "MISO", "COSMOS", "spaMultiVAE"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "adt", "Seurat_WNN", "MOFA2", "sciPENN", "moETM", "Matilda", "scMDC", "scMM",
#                 "MultiVI", "Multigrate", "TotalVI",
#                 "SMOPCA", "PRESENT", "SpatialGlue", "CellCharter", "MISO", "COSMOS", "spaMultiVAE"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True
#                 )

In [23]:
# bm.plot_gene_exp(
#     adata=new_adata,
#     genes=["CD3", "CD4", "CD8", "Cd2", "Cd4", "CD19","B220-CD45R", "IgD", "Cr2", "Ebf1", "F4-80", "CD163", "CD68","Cd24a", "Vcam1"],
#     embed=spatial,
#     figsize=(10, 7), 
#     frameon=True,
#     inner_gs_row=1,
#     inner_gs_col=1,
#     ncol=5,
#     xlabel=["CD3", "CD4", "CD8", "Cd2", "Cd4", "CD19", "B220", "IgD","Cr2", "Ebf1", "F4-80", "CD163", "CD68", "Cd24a", "Vcam1"],
#     ylabel=["T cell", "B cell", "Macrophage"],
#     only_show_left=True,
#     axis_width = 1.2,
#     axis_color="lightgrey",
#     outer_row_hspace=0.2,
#     outer_col_wspace=0.05,
#     xlabel_pad=0.012,
#     ylabel_pad=0.02,
#     background_color=lambda x: "#97a4af",
#     sizes=[12],
#     vmax="p99",
#     # vmin="p0.5",
#     color_map = "RdYlBu_r",
#     save=f"{figure_save_dir}/marker_gene_plot.pdf")

In [24]:
# sc.pl.spatial(rna, color=["Cd24a", "Mrc1", "Vcam1", "Cr2", "Ly86", "Ebf1", "Trbc1",
# "Cd8b1", "Thy1", "Cd3e", "Ccr7", "Cd5", "Cd247", "Cd8a", "Cd2", "Trbc1", "Trac", "Cd4"], spot_size=1.5, vmax="p99", ncols=3)

In [25]:
# sc.pp.normalize_total(rna)
# sc.pp.log1p(rna)
# # # sc.pp.scale(rna)

In [26]:
# new_adata = sc.concat([protein, rna], axis=1)

In [27]:
# new_adata = new_adata[:,["CD3", "CD4", "CD8", "CD19", "B220_CD45R", "IgD", "F4_80", "CD163", "CD68"]+ 
# ["Cd24a", "Vcam1", "Cr2", "Ebf1", "Cd2", "Cd4"]+ list(rna.var_names[3:100])]

In [28]:
# new_adata.var_names = [i.replace("_","-") for i in new_adata.var_names]

In [ ]:
# marker_score = bm.cal_marker_score(new_adata, label_dict=res["Cluster"], 
#                                    marker_genes=["CD3", "CD4", "CD8", "CD19", "B220-CD45R", "IgD", "F4-80", "CD163", "CD68"]+ 
# ["Cd24a", "Vcam1", "Cr2", "Ebf1", "Cd2", "Cd4"],
#                                    re_run=True,
#                                    batch_key="batch")

In [30]:
# marker_score = marker_score.drop(["rna", "adt"])

In [31]:
cmap = {
    "Seurat_WNN": "#536ea5", 
    "sciPENN": "#459f6e",
    "PRESENT": "#c5858d",
    "MOFA2": "#f59224",
    "SMOPCA": "#ef85b9",
    "moETM": "#c498c6",
    "Matilda": "#ae6f60",
    "scMDC": "#c9db34",
    "TotalVI": "#a7d38a",
    "SpatialGlue": "#1f4276",
    "CellCharter": "#d1c82e",
    "MultiVI": "#ecae4c",
    "MISO": "#f37597",
    "COSMOS": "#44bbc5",
    "spaMultiVAE": "#931b45",
    "scMM": "#e43a42",
    "Multigrate": "#8dab96"
}

In [32]:
# bm.plot_marker_score(metric_df=marker_score, xlabel=None,
#                      save=f"{figure_save_dir}/marker_score.pdf", vert=True,  label_rotation=30, figsize=(7, 3),
#                      cmap=cmap, ylabel="AUROC")

In [33]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=10,
#                 ncol=6,
#                 xlabel=[ "Seurat_WNN","MOFA2", "sciPENN",   "moETM", "Matilda",  "scMDC",
#                 "SMOPCA","PRESENT",  "SpatialGlue", "CellCharter",  "MISO","COSMOS"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=[ "Seurat_WNN",  "MOFA2", "sciPENN", "moETM", "Matilda",  "scMDC",
#                 "SMOPCA","PRESENT",  "SpatialGlue", "CellCharter", "MISO", "COSMOS"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True
#                 )

In [34]:
from benchmarker import cal_chaos_pas

In [45]:
metric = cal_chaos_pas(adata=rna, cluster_dict=res["Cluster"], batch_key=None, return_mean=True)[0]
metric = metric.drop(["rna", "adt"])

In [46]:
# y = [i[0] for i in metric["CHAOS"][:-1]]
y = [i for i in metric["CHAOS"][:-1]]

In [62]:
metric = metric.iloc[:-1,:]

In [65]:
metric = metric.reindex(["Seurat_WNN", "MOFA2", "sciPENN", "SMOPCA", "moETM", "PRESENT", "Matilda", "scMDC", "SpatialGlue", "CellCharter",
"MISO", "COSMOS", "scMM", "spaMultiVAE", "MultiVI", "Multigrate", "TotalVI"])

In [78]:
# bm.plot_bar(x=metric.index, y=list(metric['CHAOS']), orient="v", cmap=cmap,
# figsize=(5, 3), ylim=(0.925, 0.94), linewidth=0.2, ticks=[0.925, 0.93, 0.935,0.94], bar_width=0.6, rotation=30,
# save=f"{figure_save_dir}/chaos_methods.pdf")

In [79]:
# bm.plot_bar(x=metric.index, y=list(metric['PAS']), orient="v", cmap=cmap,
# figsize=(5, 3), ylim=(0.6, 1.0), linewidth=0.2, ticks=[0.6, 0.7, 0.8,0.9, 1.0], bar_width=0.6, rotation=30,
# save=f"{figure_save_dir}/pas_methods.pdf")

In [88]:
# import matplotlib.pyplot as plt
# import numpy as np

# # 1. 准备数据 (示例数据，请替换为你真实的指标数据)
# methods = ["Seurat_WNN", "MOFA2", "sciPENN", "SMOPCA", "moETM", "PRESENT", 
#            "Matilda", "scMDC", "SpatialGlue", "CellCharter", "MISO", 
#            "COSMOS", "scMM", "spaMultiVAE", "MultiVI", "Multigrate", "TotalVI"]

# # 指标 1 (对应第一张图，范围 ~0.6-1.0)
# metric1 = list(metric['CHAOS'])
# # 指标 2 (对应第二张图，范围 ~0.925-0.940)
# metric2 = list(metric['PAS'])

# x = np.arange(len(methods))

# # 2. 创建图表和左侧 Y 轴
# fig, ax1 = plt.subplots(figsize=(7.1, 3), dpi=300)

# # 绘制指标 1 (折线图 1，使用实线和圆形标记)
# color1 = '#7d96bf' # 经典蓝色
# ax1.plot(x, metric1, color=color1, marker='o', linestyle='-', linewidth=1, markersize=3, label='CHAOS')
# ax1.set_ylabel('CHAOS', color=color1, fontsize=12, fontweight='bold', labelpad=15)
# ax1.set_ylim(0.925, 0.94)

# ax1.tick_params(axis='y', labelcolor=color1)

# # 设置 X 轴标签
# ax1.set_xticks(x)
# ax1.set_xticklabels(methods, rotation=30, ha='right', fontsize=11)
# ax1.set_yticks([0.925, 0.93, 0.935, 0.94])


# # 3. 创建右侧 Y 轴
# ax2 = ax1.twinx()

# # 绘制指标 2 (折线图 2，使用虚线和方形标记以增加区分度)
# color2 = '#c2575c' # 经典红色
# ax2.plot(x, metric2, color=color2, marker='s', linestyle='--', linewidth=1, markersize=3, label='PAS')
# ax2.set_ylabel('PAS', color=color2, fontsize=12, fontweight='bold', labelpad=15)
# ax2.set_ylim(0.6, 1.0)

# ax2.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])

# ax2.tick_params(axis='y', labelcolor=color2)

# # 4. 合并图例
# lines_1, labels_1 = ax1.get_legend_handles_labels()
# lines_2, labels_2 = ax2.get_legend_handles_labels()

# # 将图例放在图内左上角 (upper left)
# ax1.legend(lines_1 + lines_2, labels_1 + labels_2, 
#            loc='upper left',   # 位置设为左上角
#            frameon=True,       # 开启图例背景框
#            framealpha=0.85,    # 设置背景为 85% 不透明度（半透明），略微透出底部的线
#            edgecolor='none',   # 去除图例边框线，视觉上更干净整洁
#            facecolor='#f8f9fa',# 设置一个极其淡的灰白色背景，增加层次感
#            fontsize=10)

# # 5. 添加网格线以辅助对齐 (可选，建议只开一个轴的网格)
# ax1.grid(True, axis='x', linestyle=':', alpha=0.6)

# # 6. 调整布局并展示
# plt.tight_layout()
# plt.savefig(f"{figure_save_dir}/chaos_pas_methods.pdf") # 保存图片
# plt.show()

In [24]:
# bm.plot_legend(category_lst=[str(i) for i in range(3)],
#                 marker="o",
#                 ncol=3,
#                 save=f"{figure_save_dir}/cluster_legend.pdf",
#                 )